In [19]:
# =========================================================
# Imports
# =========================================================
import pandas as pd
import numpy as np
import lightgbm as lgb
from pathlib import Path
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error

# =========================================================
# Paths
# =========================================================
DATA_DIR = Path("../dataset")
FEAT_DIR = Path("../features")

TRAIN_CSV = DATA_DIR / "train.csv"
TEST_CSV  = DATA_DIR / "test.csv"

TRAIN_FEAT_CSV = FEAT_DIR / "train_features.csv"
TEST_FEAT_CSV  = FEAT_DIR / "test_features.csv"

# =========================================================
# Load Base Data
# =========================================================
train_base = pd.read_csv(TRAIN_CSV, usecols=["sample_id","price"])
test_base  = pd.read_csv(TEST_CSV,  usecols=["sample_id"])

train_struct = pd.read_csv(TRAIN_FEAT_CSV)
test_struct  = pd.read_csv(TEST_FEAT_CSV)

print("train_struct:", train_struct.shape, "test_struct:", test_struct.shape)

# Reindex to match canonical order
train_struct = (
    train_struct.set_index("sample_id")
                .reindex(train_base["sample_id"])
                .reset_index()
)
test_struct = (
    test_struct.set_index("sample_id")
               .reindex(test_base["sample_id"])
               .reset_index()
)

# Sanity checks
assert (train_struct["sample_id"].values == train_base["sample_id"].values).all()
assert (test_struct["sample_id"].values  == test_base["sample_id"].values).all()

# =========================================================
# Load Features
# =========================================================
X_tfidf_train = np.load(FEAT_DIR / "tfidf_train.npy")
X_tfidf_test  = np.load(FEAT_DIR / "tfidf_test.npy")

train_img = np.load(FEAT_DIR / "train_img.npy")
test_img  = np.load(FEAT_DIR / "test_img.npy")

print("TF-IDF:", X_tfidf_train.shape, X_tfidf_test.shape)
print("IMG:", train_img.shape, test_img.shape)

# Align checks
assert len(train_img) == len(train_struct) == len(train_base)
assert len(test_img)  == len(test_struct)  == len(test_base)

# =========================================================
# Final Feature Matrices
# =========================================================
struct_cols = [c for c in train_struct.columns if c != "sample_id"]

X_struct_train = train_struct[struct_cols].fillna(0).to_numpy()
X_struct_test  = test_struct[struct_cols].fillna(0).to_numpy()

X_train = np.hstack([X_struct_train, X_tfidf_train, train_img])
X_test  = np.hstack([X_struct_test,  X_tfidf_test,  test_img])

y = np.log1p(train_base["price"].values)

print("Final train:", X_train.shape)
print("Final test:", X_test.shape)

# =========================================================
# SMAPE Metric
# =========================================================
def smape(y_true, y_pred):
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2
    diff = np.abs(y_true - y_pred) / np.where(denom==0, 1, denom)
    diff[denom == 0] = 0
    return np.mean(diff) * 100

# =========================================================
# CV Setup
# =========================================================
kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(y))
test_preds = np.zeros(len(test_base))

# =========================================================
# Tuned LightGBM Params
# =========================================================
lgb_params = {
    'objective': 'regression',
    'metric': 'mae',
    'boosting_type': 'gbdt',
    'learning_rate': 0.025,      # lower LR
    'num_leaves': 128,           # larger trees
    'max_depth': 12,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'min_data_in_leaf': 75,      # more regularization
    'lambda_l1': 0.2,
    'lambda_l2': 0.5,
    'seed': 42,
    'verbose': -1
}

# =========================================================
# Training Loop
# =========================================================
for fold, (tr_idx, val_idx) in enumerate(kf.split(X_train)):
    print(f"\n===== Fold {fold+1} =====")
    dtrain = lgb.Dataset(X_train[tr_idx], label=y[tr_idx])
    dvalid = lgb.Dataset(X_train[val_idx], label=y[val_idx])

    model = lgb.train(
        lgb_params, dtrain,
        valid_sets=[dtrain, dvalid],
        num_boost_round=8000,
        callbacks=[
            lgb.early_stopping(stopping_rounds=200),
            lgb.log_evaluation(200)
        ]
    )

    oof_preds[val_idx] = model.predict(X_train[val_idx], num_iteration=model.best_iteration)
    test_preds += model.predict(X_test, num_iteration=model.best_iteration) / kf.n_splits

# =========================================================
# Evaluation
# =========================================================
mae_log  = mean_absolute_error(y, oof_preds)
mae_orig = mean_absolute_error(np.expm1(y), np.expm1(oof_preds))
smape_cv = smape(np.expm1(y), np.expm1(oof_preds))

print("\n===== CV Results =====")
print("CV MAE (log):", mae_log)
print("CV MAE (orig):", mae_orig)
print("CV SMAPE:", smape_cv)

# =========================================================
# Save Submission
# =========================================================
pred_final = np.expm1(test_preds)

# Clip extreme predictions
lo, hi = np.percentile(train_base["price"].values, [0.5, 99.5])
pred_final = np.clip(pred_final, lo, hi)

sub = pd.DataFrame({"sample_id": test_base["sample_id"], "price": pred_final})
sub.to_csv("submission_lgb_tuned.csv", index=False)
print("Saved submission_lgb_tuned.csv ✅")


train_struct: (75000, 12) test_struct: (75000, 12)
TF-IDF: (75000, 200) (75000, 200)
IMG: (75000, 256) (75000, 256)
Final train: (75000, 467)
Final test: (75000, 467)

===== Fold 1 =====
Training until validation scores don't improve for 200 rounds
[200]	training's l1: 0.462703	valid_1's l1: 0.551806
[400]	training's l1: 0.382844	valid_1's l1: 0.537682
[600]	training's l1: 0.324968	valid_1's l1: 0.531427
[800]	training's l1: 0.279151	valid_1's l1: 0.527468
[1000]	training's l1: 0.241302	valid_1's l1: 0.525016
[1200]	training's l1: 0.209493	valid_1's l1: 0.523305
[1400]	training's l1: 0.182379	valid_1's l1: 0.521958
[1600]	training's l1: 0.159248	valid_1's l1: 0.52081
[1800]	training's l1: 0.139385	valid_1's l1: 0.519893
[2000]	training's l1: 0.122223	valid_1's l1: 0.519329
[2200]	training's l1: 0.107389	valid_1's l1: 0.518629
[2400]	training's l1: 0.0944161	valid_1's l1: 0.518123
[2600]	training's l1: 0.0833476	valid_1's l1: 0.517653
[2800]	training's l1: 0.0736598	valid_1's l1: 0.5173